In [37]:
import os
import pandas as pd

HOUSING_PATH = "./"

def save_fig(fig_id, tight_layout=True, fig_extension="png", resolution=300):
    path = os.path.join("./", fig_id + "." + fig_extension)
    print("Saving figure", fig_id)
    if tight_layout:
        plt.tight_layout()
    plt.savefig(path, format=fig_extension, dpi=resolution)

def load_housing_data(housing_path=HOUSING_PATH):
    csv_path = os.path.join(housing_path, "housing.csv")
    return pd.read_csv(csv_path)

housing = load_housing_data()

def split_train_test_by_lib():
    from sklearn.model_selection import train_test_split
    return train_test_split(housing, test_size=0.2, random_state=42)

def limit_test():
    import numpy as np
    #housing["median_income"].hist()
    housing["income_cat"] = np.ceil(housing["median_income"] / 1.5)  # Divide by 1.5 to limit the number of income categories
    #housing["income_cat"].where(housing["income_cat"] < 5, 5.0, inplace=True)    # Label those above 5 as 5
    housing["income_cat"] = pd.cut(housing["median_income"],
                               bins=[0., 1.5, 3.0, 4.5, 6., np.inf],
                               labels=[1, 2, 3, 4, 5])
    #housing["income_cat"].value_counts()
    #housing["income_cat"].hist()

def fenceng_split():
    from sklearn.model_selection import StratifiedShuffleSplit
    split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    for train_index, test_index in split.split(housing, housing["income_cat"]):
        strat_train_set = housing.loc[train_index]
        strat_test_set = housing.loc[test_index]       
    return strat_train_set, strat_test_set

limit_test()
train_set, test_set = split_train_test_by_lib()
strat_train_set, strat_test_set = fenceng_split()
#print(strat_train_set)
#print(strat_test_set)

housing = strat_train_set.drop("median_house_value", axis=1) # drop labels for training set
housing_labels = strat_train_set["median_house_value"].copy()
#print(housing)
#print(housing_labels)

sample_incomplete_rows = housing[housing.isnull().any(axis=1)].head()   #total_bedrooms有一些缺失值
sample_incomplete_rows


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity,income_cat
1606,-122.08,37.88,26.0,2947.0,NaN,825.0,626.0,2.9330,NEAR BAY,2
10915,-117.87,33.73,45.0,2264.0,NaN,1970.0,499.0,3.4193,<1H OCEAN,3
19150,-122.70,38.35,14.0,2313.0,NaN,954.0,397.0,3.7813,<1H OCEAN,3
4186,-118.23,34.13,48.0,1308.0,NaN,835.0,294.0,4.2891,<1H OCEAN,3
16885,-122.40,37.58,26.0,3281.0,NaN,1145.0,480.0,6.3580,NEAR OCEAN,5


In [38]:
#sample_incomplete_rows.dropna(subset=["total_bedrooms"])    # 去掉total_bedrooms的所有项

In [43]:
#sample_incomplete_rows.drop("total_bedrooms", axis=1)       # 去掉整个属性

In [44]:
median = housing["total_bedrooms"].median()
sample_incomplete_rows["total_bedrooms"].fillna(median, inplace=True) # 进行赋值（0、平均值、中位数等等）
sample_incomplete_rows

/tmp/ipykernel_7346/3955344275.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  sample_incomplete_rows["total_bedrooms"].fillna(median, inplace=True) # 进行赋值（0、平均值、中位数等等）


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity,income_cat
1606,-122.08,37.88,26.0,2947.0,433.0,825.0,626.0,2.9330,NEAR BAY,2
10915,-117.87,33.73,45.0,2264.0,433.0,1970.0,499.0,3.4193,<1H OCEAN,3
19150,-122.70,38.35,14.0,2313.0,433.0,954.0,397.0,3.7813,<1H OCEAN,3
4186,-118.23,34.13,48.0,1308.0,433.0,835.0,294.0,4.2891,<1H OCEAN,3
16885,-122.40,37.58,26.0,3281.0,433.0,1145.0,480.0,6.3580,NEAR OCEAN,5


In [45]:
try:
    from sklearn.impute import SimpleImputer # Scikit-Learn 0.20+
except ImportError:
    from sklearn.preprocessing import Imputer as SimpleImputer

imputer = SimpleImputer(strategy="median")    #指定用某属性的中位数来替换该属性所有的缺失值
housing_num = housing.drop("ocean_proximity", axis=1)  #只有数值属性才能算出中位数，我们需要创建一份不包括文本属性ocean_proximity的数据副本：
imputer.fit(housing_num)   #用fit()方法将imputer实例拟合到训练数据

imputer.statistics_ #imputer计算出了每个属性的中位数，并将结果保存在了实例变量statistics_中

array([-118.51   ,   34.26   ,   29.     , 2119.     ,  433.     ,
       1164.     ,  408.     ,    3.54155,    3.     ])

In [47]:
X = imputer.transform(housing_num)  #用这个“训练过的”imputer来对训练集进行转换，将缺失值替换为中位数  结果是一个包含转换后特征的普通的 Numpy 数组。
housing_tr = pd.DataFrame(X, columns=housing_num.columns) #将其放回到 PandasDataFrame中
#housing_tr.loc[sample_incomplete_rows.index.values]

In [48]:
housing_cat = housing[['ocean_proximity']]
housing_cat.head(10)
#4 NEAR OCEAN：直接靠近海洋的区域，可能包括海滩和沿海城镇。
#3 NEAR BAY：靠近海湾的地区，虽然海湾是海洋的一部分，但通常距离海洋稍远一些。
#2 ISLAND
#1 INLAND
#0 <1H OCEAN：虽然距离海洋少于1小时，但不一定直接接触海洋，可能位于更远的内陆区域。

,ocean_proximity
12655,INLAND
15502,NEAR OCEAN
2908,INLAND
14053,NEAR OCEAN
20496,<1H OCEAN
1481,NEAR BAY
18125,<1H OCEAN
5830,<1H OCEAN
17989,<1H OCEAN
4861,<1H OCEAN


In [50]:
try:
    from sklearn.preprocessing import OrdinalEncoder
except ImportError:
    from future_encoders import OrdinalEncoder # Scikit-Learn < 0.20

ordinal_encoder = OrdinalEncoder()
housing_cat_encoded = ordinal_encoder.fit_transform(housing_cat)
housing_cat_encoded[:10]

array([[1.],
       [4.],
       [1.],
       [4.],
       [0.],
       [3.],
       [0.],
       [0.],
       [0.],
       [0.]])

In [51]:
ordinal_encoder.categories_

[array(['<1H OCEAN', 'INLAND', 'ISLAND', 'NEAR BAY', 'NEAR OCEAN'],
       dtype=object)]

In [52]:
try:
    from sklearn.preprocessing import OrdinalEncoder # just to raise an ImportError if Scikit-Learn < 0.20
    from sklearn.preprocessing import OneHotEncoder
except ImportError:
    from future_encoders import OneHotEncoder # Scikit-Learn < 0.20

cat_encoder = OneHotEncoder()
housing_cat_1hot = cat_encoder.fit_transform(housing_cat)
housing_cat_1hot #结果是一个 SciPy 稀疏矩阵

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 16512 stored elements and shape (16512, 5)>

In [53]:
housing_cat_1hot.toarray()  # NumPy 数组 

array([[0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 1.],
       [0., 1., 0., 0., 0.],
       ...,
       [1., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0.]])